# Explore Meta's pretrained Byte Latent Transformer

This notebook defaults to `facebook/blt-1b` and its nested entropy model, loading them once and keeping them resident on one MI300A GPU for repeated experiments. Set `BLT_REPO = "facebook/blt-7b"` in the loading cell to explore the larger checkpoint; it has also been validated on one MI300A.

Before running it:

1. Activate `$WRKSPC/tuolumne_conda_291_643_blt` before requesting the compute-node allocation.
2. Launch or attach the notebook kernel from inside that allocation.
3. Select that conda environment as the notebook's Python kernel.

The notebook intentionally uses only GPU 0, even if the allocation exposes four GPUs. The first model call can spend roughly 30 seconds compiling FlexAttention kernels. Native BLT generation does not use a KV cache, so start with short completions while exploring.

## 1. Initialize the compute-node runtime

BLT's generation helper expects a process group even for one GPU. This cell creates an isolated, single-process RCCL group without relying on launcher rank variables.

In [ ]:
from __future__ import annotations

import atexit
import math
import os
import shutil
import socket
import sys
import tempfile
import time
from pathlib import Path
from pprint import pprint

REPO_ROOT = Path("/p/vast1/kirchenb/hlm-root/blt")
CACHE_DIR = Path("/p/vast1/kirchenb/hlm-root/hf-cache/hub")
if not (REPO_ROOT / "bytelatent").is_dir():
    raise RuntimeError(f"BLT checkout not found at {REPO_ROOT}")

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Keep the ROCm-safe BlockMask metadata path diagnosed by the smoke tests.
os.environ.setdefault("BLT_COMPILE_BLOCK_MASK", "0")
os.environ.setdefault("HF_HUB_CACHE", str(CACHE_DIR))

import torch
import torch.distributed as dist

if not torch.cuda.is_available():
    raise RuntimeError("No ROCm GPU is visible. Start this kernel inside a compute allocation.")

GPU_INDEX = 0
torch.cuda.set_device(GPU_INDEX)
DEVICE = torch.device("cuda", GPU_INDEX)
_RDZV_DIR: str | None = globals().get("_RDZV_DIR")

if not dist.is_initialized():
    _RDZV_DIR = tempfile.mkdtemp(prefix="blt-notebook-rdzv-")
    rendezvous_file = Path(_RDZV_DIR) / "store"
    dist.init_process_group(
        backend="nccl",
        init_method=f"file://{rendezvous_file}",
        rank=0,
        world_size=1,
    )
elif dist.get_world_size() != 1:
    raise RuntimeError("This interactive notebook expects a one-process process group.")

def shutdown_blt_runtime() -> None:
    """Release the process group and its local rendezvous directory."""
    if dist.is_initialized():
        dist.destroy_process_group()
    if _RDZV_DIR is not None:
        shutil.rmtree(_RDZV_DIR, ignore_errors=True)

atexit.register(shutdown_blt_runtime)

properties = torch.cuda.get_device_properties(DEVICE)
pprint(
    {
        "hostname": socket.gethostname(),
        "torch": torch.__version__,
        "hip": torch.version.hip,
        "device": properties.name,
        "device_memory_gib": round(properties.total_memory / 2**30, 1),
        "visible_gpus": torch.cuda.device_count(),
        "process_group_world_size": dist.get_world_size(),
        "repo_root": str(REPO_ROOT),
        "cache_dir": str(CACHE_DIR),
    }
)

## 2. Load the checkpoint once

The official BLT repository contains both the 1B-parameter BLT checkpoint and the entropy model used to choose dynamic byte patches. Re-running this cell will reuse the existing Python objects rather than load a second copy onto the GPU.

Set `LOCAL_FILES_ONLY = True` if you want to require that every artifact already exists in the shared cache.

In [ ]:
# Edit this value directly in the notebook, or set BLT_REPO in the kernel environment.
BLT_REPO = os.environ.get("BLT_REPO", "facebook/blt-1b")
LOCAL_FILES_ONLY = False
CACHE_DIR.mkdir(parents=True, exist_ok=True)

from huggingface_hub import hf_hub_download

from bytelatent.data.patcher import to_device
from bytelatent.entropy_model import load_entropy_model
from bytelatent.generate_blt import generate_nocache
from bytelatent.hf import BltTokenizerAndPatcher
from bytelatent.model.blt import ByteLatentTransformer
from bytelatent.tokenizers.blt_tokenizer import BltTokenizer
from bytelatent.tokenizers.constants import BOE_ID, BOS_ID, BPE_ID, EOS_ID, OFFSET, PAD_ID

def load_pretrained_blt():
    torch.cuda.reset_peak_memory_stats(GPU_INDEX)
    started = time.monotonic()

    entropy_state = hf_hub_download(
        BLT_REPO,
        "entropy_model/consolidated.pth",
        cache_dir=str(CACHE_DIR),
        local_files_only=LOCAL_FILES_ONLY,
    )
    hf_hub_download(
        BLT_REPO,
        "entropy_model/params.json",
        cache_dir=str(CACHE_DIR),
        local_files_only=LOCAL_FILES_ONLY,
    )
    entropy_model, _ = load_entropy_model(
        os.path.dirname(entropy_state),
        entropy_state,
    )

    model = ByteLatentTransformer.from_pretrained(
        BLT_REPO,
        cache_dir=str(CACHE_DIR),
        local_files_only=LOCAL_FILES_ONLY,
    )
    tok_and_patcher = BltTokenizerAndPatcher.from_pretrained(
        BLT_REPO,
        cache_dir=str(CACHE_DIR),
        local_files_only=LOCAL_FILES_ONLY,
    )

    tokenizer = tok_and_patcher.tokenizer_args.build()
    if not isinstance(tokenizer, BltTokenizer):
        raise TypeError(f"Expected BltTokenizer, got {type(tokenizer).__name__}")
    patcher = tok_and_patcher.patcher_args.build()

    dtype = {
        "fp32": torch.float32,
        "fp16": torch.float16,
        "bf16": torch.bfloat16,
    }[tok_and_patcher.distributed_args.model_dtype]
    model = model.to(device=DEVICE, dtype=dtype).eval()
    patcher.realtime_patching = True
    patcher.entropy_model, patching_device = to_device(
        entropy_model,
        tok_and_patcher.patcher_args.patching_device,
    )
    patcher.entropy_model.eval()

    torch.cuda.synchronize()
    metadata = {
        "repo": BLT_REPO,
        "dtype": str(dtype),
        "patching_mode": str(patcher.patching_mode),
        "patching_threshold": patcher.threshold,
        "patching_device": str(patching_device),
        "load_seconds": round(time.monotonic() - started, 2),
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "allocated_gib": round(torch.cuda.memory_allocated(GPU_INDEX) / 2**30, 3),
        "peak_gib": round(torch.cuda.max_memory_allocated(GPU_INDEX) / 2**30, 3),
    }
    return model, tokenizer, patcher, metadata

if "model" not in globals():
    model, tokenizer, patcher, model_metadata = load_pretrained_blt()
else:
    print("Model objects already exist; skipping reload.")

pprint(model_metadata)

## 3. Inspect entropy-selected byte patches

BLT encodes UTF-8 bytes rather than subword tokens. The small entropy model decides where patches begin; the local encoder converts bytes within each patch into representations for the global transformer. `<NEXT>` is the extra future position required by BLT's generation-time patch calculation.

A patch can split a multibyte Unicode character. When that happens, the display deliberately shows escaped byte values instead of hiding the boundary.

In [ ]:
SPECIAL_TOKEN_NAMES = {
    BOE_ID: "<BOE>",
    BOS_ID: "<BOS>",
    EOS_ID: "<EOS>",
    PAD_ID: "<PAD>",
    BPE_ID: "<BPE>",
}

def render_token_span(token_ids: list[int | None]) -> str:
    pieces: list[str] = []
    byte_buffer: list[int] = []

    def flush_bytes() -> None:
        if byte_buffer:
            pieces.append(bytes(byte_buffer).decode("utf-8", errors="backslashreplace"))
            byte_buffer.clear()

    for token_id in token_ids:
        if token_id is None:
            flush_bytes()
            pieces.append("<NEXT>")
        elif token_id in SPECIAL_TOKEN_NAMES:
            flush_bytes()
            pieces.append(SPECIAL_TOKEN_NAMES[token_id])
        elif OFFSET <= token_id < OFFSET + 256:
            byte_buffer.append(token_id - OFFSET)
        else:
            flush_bytes()
            pieces.append(f"<ID:{token_id}>")
    flush_bytes()
    return "".join(pieces)

@torch.inference_mode()
def inspect_patches(prompt: str, *, show_entropies: bool = False) -> dict:
    token_ids = tokenizer.encode(prompt, add_eos=False)
    tokens = torch.tensor(token_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    patch_lengths, entropies = patcher.patch(tokens, include_next_token=True)
    lengths = patch_lengths[0][patch_lengths[0] > 0].tolist()

    extended_ids: list[int | None] = [*token_ids, None]
    patches = []
    start = 0
    for patch_index, length in enumerate(lengths):
        stop = start + length
        span_ids = extended_ids[start:stop]
        patches.append(
            {
                "patch": patch_index,
                "positions": f"{start}:{stop}",
                "length": length,
                "content": render_token_span(span_ids),
            }
        )
        start = stop

    entropy_values = [] if entropies is None else entropies[0].float().cpu().tolist()
    summary = {
        "prompt": prompt,
        "utf8_bytes": len(prompt.encode("utf-8")),
        "input_positions_including_bos": len(token_ids),
        "patch_count": len(lengths),
        "patch_lengths": lengths,
        "mean_patch_length": round(sum(lengths) / len(lengths), 3),
        "entropy_min": round(min(entropy_values), 4) if entropy_values else None,
        "entropy_mean": round(sum(entropy_values) / len(entropy_values), 4) if entropy_values else None,
        "entropy_max": round(max(entropy_values), 4) if entropy_values else None,
    }

    pprint(summary)
    print("\nPatches:")
    for row in patches:
        print(
            f"  {row['patch']:>2}  positions={row['positions']:<9} "
            f"length={row['length']:<3} {row['content']!r}"
        )
    if show_entropies and entropy_values:
        print("\nEntropy score by input position:")
        for position, (token_id, value) in enumerate(zip(token_ids, entropy_values)):
            print(
                f"  {position:>3}  entropy={value:>7.4f}  "
                f"token={render_token_span([token_id])!r}"
            )
    return {"summary": summary, "patches": patches, "entropies": entropy_values}

patch_details = inspect_patches("Café naïve 日本語 — byte-level models!", show_entropies=False)

## 4. Generate repeatedly without reloading

`generate_text` accepts either one prompt or a list of differently sized prompts. Greedy decoding is deterministic. Set `sample=True` to explore temperature, top-k, or nucleus sampling.

The upstream native generator always runs exactly `max_gen_len` byte steps and recomputes the sequence without a KV cache. The helper decodes only through the first generated EOS marker.

In [ ]:
@torch.inference_mode()
def generate_text(
    prompts: str | list[str],
    *,
    max_gen_len: int = 24,
    max_prompt_len: int = 256,
    sample: bool = False,
    temperature: float = 1.0,
    top_k: int = 0,
    top_p: float = 0.0,
    seed: int = 1234,
) -> list[dict]:
    if isinstance(prompts, str):
        prompts = [prompts]
    if not prompts or not all(isinstance(prompt, str) for prompt in prompts):
        raise ValueError("Provide one prompt string or a non-empty list of strings.")
    if max_gen_len < 1:
        raise ValueError("max_gen_len must be positive.")
    if sample and temperature <= 0:
        raise ValueError("Sampling temperature must be positive.")

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.cuda.reset_peak_memory_stats(GPU_INDEX)
    torch.cuda.synchronize()
    started = time.monotonic()
    output_ids = generate_nocache(
        prompts,
        model=model,
        tokenizer=tokenizer,
        patcher=patcher,
        max_prompt_len=max_prompt_len,
        max_gen_len=max_gen_len,
        use_sampling=sample,
        temp=temperature,
        top_k=top_k,
        top_p=top_p,
    )
    torch.cuda.synchronize()
    elapsed = time.monotonic() - started

    results = []
    for prompt, token_ids in zip(prompts, output_ids):
        completion = tokenizer.decode(token_ids, cut_at_eos=True)
        record = {
            "prompt": prompt,
            "completion": completion,
            "full_text": prompt + completion,
            "generated_token_ids": token_ids,
            "generated_eos": tokenizer.eos_id in token_ids,
        }
        results.append(record)
        print(f"PROMPT:     {prompt!r}")
        print(f"COMPLETION: {completion!r}\n")

    print(
        f"batch={len(prompts)} bytes_per_prompt={max_gen_len} "
        f"seconds={elapsed:.3f} "
        f"peak_memory_gib={torch.cuda.max_memory_allocated(GPU_INDEX) / 2**30:.3f}"
    )
    return results

# Edit this prompt and re-run only this cell as often as you like.
results = generate_text("The capital of France is", max_gen_len=16)
results

### Batched prompts

Uncomment the call after editing the examples. Variable-length batching uses the eager BlockMask path validated by the ROCm smoke tests.

In [ ]:
batch_prompts = [
    "def fibonacci(n):\n    ",
    "Café naïve 日本語 — ",
    "Ths sentnce has typoss!!! ",
]

# batch_results = generate_text(batch_prompts, max_gen_len=16)
# batch_results

### Sampling

For top-p sampling, leave `top_k=0`. For top-k sampling, leave `top_p=0.0`. Reusing the same seed makes an experiment repeatable.

In [ ]:
# sampled = generate_text(
#     "Once upon a time",
#     max_gen_len=32,
#     sample=True,
#     temperature=0.8,
#     top_p=0.9,
#     seed=42,
# )
# sampled

## 5. Score complete strings

This computes teacher-forced negative log-likelihood and bits per UTF-8 byte (BPB), a natural metric for a byte-level model. It is useful for comparing alternate spellings, code fragments, languages, or noisy text without generating.

In [ ]:
import torch.nn.functional as F

@torch.inference_mode()
def score_text(text: str) -> dict:
    encoded = tokenizer.encode(text, add_bos=True, add_eos=True)
    tokens = torch.tensor(encoded, dtype=torch.long, device=DEVICE).unsqueeze(0)
    inputs = tokens[:, :-1]
    targets = tokens[:, 1:]
    patch_lengths, _ = patcher.patch(inputs, include_next_token=True)
    logits = model(inputs, patch_lengths=patch_lengths)
    nll = F.cross_entropy(
        logits.flatten(0, 1).float(),
        targets.flatten(0, 1),
        reduction="sum",
    ).item()
    predicted_tokens = targets.numel()
    nonzero_lengths = patch_lengths[0][patch_lengths[0] > 0].tolist()
    record = {
        "text": text,
        "utf8_bytes": len(text.encode("utf-8")),
        "predicted_positions_including_eos": predicted_tokens,
        "patch_count": len(nonzero_lengths),
        "mean_patch_length": round(sum(nonzero_lengths) / len(nonzero_lengths), 3),
        "nll": nll,
        "bits_per_predicted_byte": nll / math.log(2) / predicted_tokens,
    }
    pprint(record)
    return record

# score = score_text("BLT predicts UTF-8 bytes directly.")
# score

## Optional cleanup

Stopping or restarting the kernel releases everything automatically. To release the process group and GPU objects without stopping the kernel, uncomment the lines below. Re-run the initialization and loading cells before doing more inference.

In [ ]:
# del model, tokenizer, patcher
# torch.cuda.empty_cache()
# shutdown_blt_runtime()